# Lab 21 — Fine-tuning LLMs · RUN ALL (T4)

Chạy từ trên xuống. **Runtime > Change runtime type > T4 GPU** trước khi bắt đầu.

| Ô | Làm gì | Thời gian |
|---|---|---|
| 1 | clone + install | ~1 phút |
| 2 | smoke: import + unit test | ~30 giây |
| 3 | **pipeline NB1 -> NB5 + NB6** | ~80–110 phút |
| 4 | gatekeeper + in kết quả + tải zip về | ~10 giây |


In [ ]:
# @title 1. Setup — clone + install (chạy ô này trước)
import os, subprocess, sys

REPO = "https://github.com/HK-204/Day21-Track3-DaoDucHai-2A202602752-Finetuning-Lab.git" #@param {type:"string"}
repo_name = REPO.rstrip("/").split("/")[-1].replace(".git", "")
if not os.path.exists(repo_name):
    subprocess.run(["git", "clone", "-q", REPO], check=True)
os.chdir(f"/content/{repo_name}")
subprocess.run(["git", "pull", "-q"], check=False)
sys.path.insert(0, "src")

# Install from requirements.txt, NOT a copied list.
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"],
               check=True)

import torch
print("commit :", subprocess.run(["git","rev-parse","--short","HEAD"],
                                 capture_output=True, text=True).stdout.strip())
print("GPU    :", torch.cuda.get_device_name(0) if torch.cuda.is_available()
      else "NONE — Runtime > Change runtime type > T4 GPU")
if torch.cuda.is_available():
    print("VRAM   : %.1f GB" % (torch.cuda.get_device_properties(0).total_memory/1024**3))


In [ ]:
# @title 2. Smoke — imports, seed data, unit tests (no GPU needed)
!python scripts/verify.py --smoke


In [ ]:
# @title 3. Core pipeline — NB1 → NB5 (+ tuỳ chọn NB6)
# EVAL_LIMIT: để trống ("") để chạy full 50 mẫu (bắt buộc khi nộp bài).
# STAGES: "nb1 nb2 nb3 nb4 nb5 nb6" chạy đủ core và phần thưởng NB6.
import os
COMPUTE_TIER = "T4"        # @param ["CPU","LAPTOP","T4","BIGGPU"]
EVAL_LIMIT   = ""          # @param ["", "4", "8", "16", "25"]
STAGES       = "nb1 nb2 nb3 nb4 nb5 nb6"   # @param {type:"string"}

os.environ["COMPUTE_TIER"] = COMPUTE_TIER
if EVAL_LIMIT:
    os.environ["EVAL_LIMIT"] = EVAL_LIMIT
else:
    os.environ.pop("EVAL_LIMIT", None)

from labkit import device
print(device.banner(), "\n")

!python scripts/colab_run.py {STAGES}


In [ ]:
# @title 4. Gatekeeper + results + Tải artefacts về máy
!python scripts/verify.py
print("\n================ results/ ================")
!ls -la results/
!echo && echo "---- runs.csv ----" && cat results/runs.csv 2>/dev/null
!echo && echo "---- verdict.json ----" && cat results/verdict.json 2>/dev/null

# Đóng gói kết quả và adapter correct để tải về nộp bài / điền report
!zip -q -r /content/lab21_results.zip results/ adapters/correct/
try:
    from google.colab import files
    files.download('/content/lab21_results.zip')
except Exception as e:
    print("Download error or running outside Colab:", e)
